# 02 — Exploratory Data Analysis
**HHE Lab Sardinia · Marine Litter Hazard Assessment**

Inputs → `data/processed/beach_litter.csv` · `floating_litter.csv`  
Outputs → `data/figures/*.png`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from pathlib import Path

ROOT  = Path("..").resolve()
OUT   = ROOT / "data" / "processed"
FIGS  = ROOT / "data" / "figures"
FIGS.mkdir(exist_ok=True)

beach    = pd.read_csv(OUT / "beach_litter.csv")
floating = pd.read_csv(OUT / "floating_litter.csv")

sns.set_theme(style="whitegrid", font_scale=1.1)

BEACH_COLORS = {
    "MWE_SAR_1": "#e74c3c", "MWE_SAR_2": "#3498db",
    "MWE_SAR_3": "#2ecc71", "MWE_SAR_4": "#f39c12",
    "MWE_SAR_5": "#9b59b6", "MWE_SAR_6": "#1abc9c"
}
BEACH_NAMES = {
    "MWE_SAR_1": "Alghero Lido", "MWE_SAR_2": "Cagliari Poetto",
    "MWE_SAR_3": "Castiadas Costa Rei", "MWE_SAR_4": "Oristano Is Arenas",
    "MWE_SAR_5": "San Teodoro La Cinta", "MWE_SAR_6": "S.A. Arresi Porto Pino"
}
print("Data loaded.")
print(f"Beach: {len(beach)} surveys · Floating: {len(floating)} obs")

---
## 1. Beach Litter — Trend per stazione

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
yearly = beach.groupby(["beach_id","year"])["items_per_100m"].mean().reset_index()
for bid, grp in yearly.groupby("beach_id"):
    ax.plot(grp["year"], grp["items_per_100m"],
            marker="o", linewidth=2,
            color=BEACH_COLORS.get(bid, "#666"),
            label=BEACH_NAMES.get(bid, bid))
ax.axhline(150, color="gray", linestyle="--", linewidth=1.2, alpha=0.7,
           label="EU indicative threshold (150 items/100m)")
ax.set_title("Beach Litter — Trend per spiaggia (items/100m)", fontweight="bold", pad=12)
ax.set_xlabel("Year"); ax.set_ylabel("Mean items / 100m")
ax.set_xticks(yearly["year"].unique())
ax.legend(loc="upper right", framealpha=0.9, fontsize=9)
plt.tight_layout()
plt.savefig(FIGS / "beach_trend_per_station.png", dpi=150)
plt.show()

## 2. Beach Litter — Distribuzione per anno

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
years_sorted = sorted(beach["year"].unique())
data_by_year = [beach[beach["year"]==y]["items_per_100m"].dropna().values for y in years_sorted]
bp = ax.boxplot(data_by_year, tick_labels=years_sorted, patch_artist=True,
                medianprops=dict(color="black", linewidth=2))
for patch, color in zip(bp["boxes"], ["#74b9ff","#a29bfe","#fd79a8","#fdcb6e"]):
    patch.set_facecolor(color); patch.set_alpha(0.85)
ax.axhline(150, color="gray", linestyle="--", linewidth=1.2, alpha=0.7)
ax.set_title("Distribuzione items/100m per anno — tutte le spiagge", fontweight="bold", pad=12)
ax.set_xlabel("Year"); ax.set_ylabel("items / 100m")
plt.tight_layout()
plt.savefig(FIGS / "beach_boxplot_year.png", dpi=150)
plt.show()

## 3. Beach Litter — Top categorie (JointListCategory)

In [ ]:
# Reload item-level data with categories
MOD4_DIR = ROOT / "Labenv" / "Modulo_4_2020_Med_Occ_2018-2023"
RIFIUTI_RENAME = {
    "SampleID":"survey_id","CodiceCampionamento":"survey_id",
    "JointListCategory":"category","IDCategoriaRifiuto":"category",
    "NumeroOggetti":"n_items","NumeroItems":"n_items","Sorgente":"source"
}
CAMP_RENAME2 = {
    "SampleID":"survey_id","CodiceCampionamento":"survey_id",
    "Year":"year","Anno":"year","CodiceSpiaggia":"beach_id"
}
def norm(df, rename, keep):
    df = df.rename(columns={k:v for k,v in rename.items() if k in df.columns})
    return df[[c for c in keep if c in df.columns]]

rif_frames = []
for f in sorted(MOD4_DIR.glob("*.xl*")):
    engine = "xlrd" if f.suffix==".xls" else "openpyxl"
    try:
        xl   = pd.ExcelFile(f, engine=engine)
        rif  = norm(xl.parse("RifiutiCamp"), RIFIUTI_RENAME, ["survey_id","category","n_items","source"])
        camp = norm(xl.parse("SpiaggiaCamp"), CAMP_RENAME2, ["survey_id","beach_id","year"])
        rif_frames.append(rif.merge(camp, on="survey_id", how="left"))
    except: pass

rif_df = pd.concat(rif_frames, ignore_index=True)
rif_df["n_items"] = pd.to_numeric(rif_df["n_items"], errors="coerce").fillna(0)
cat_total = rif_df.groupby("category")["n_items"].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
top10 = cat_total.head(10)
bars = ax.barh(top10.index[::-1], top10.values[::-1], color="#3498db", alpha=0.85)
for bar, val in zip(bars, top10.values[::-1]):
    ax.text(val + top10.max()*0.01, bar.get_y()+bar.get_height()/2,
            f"{int(val):,}", va="center", fontsize=9)
ax.set_title("Top 10 categorie rifiuti spiaggiati (2020–2023)", fontweight="bold", pad=12)
ax.set_xlabel("Total items")
plt.tight_layout()
plt.savefig(FIGS / "beach_top_categories.png", dpi=150)
plt.show()

## 4. Beach Litter — Composizione per anno (top 5)

In [ ]:
top5_cats = cat_total.head(5).index.tolist()
cat_year = (rif_df[rif_df["category"].isin(top5_cats)]
            .groupby(["year","category"])["n_items"].sum()
            .unstack(fill_value=0))

fig, ax = plt.subplots(figsize=(9, 5))
cat_year.plot(kind="bar", stacked=True, ax=ax, colormap="tab10", alpha=0.85, width=0.6)
ax.set_title("Composizione rifiuti per anno (top 5 categorie)", fontweight="bold", pad=12)
ax.set_xlabel("Year"); ax.set_ylabel("Total items")
ax.legend(title="Category", loc="upper left", fontsize=8, framealpha=0.9)
ax.set_xticklabels(cat_year.index.astype(int), rotation=0)
plt.tight_layout()
plt.savefig(FIGS / "beach_category_by_year.png", dpi=150)
plt.show()

## 5. Beach Litter — Source attribution

In [ ]:
src = rif_df.groupby("source")["n_items"].sum().dropna().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(9, 4))
src.plot(kind="bar", ax=ax, color="#e67e22", alpha=0.85, width=0.5)
ax.set_title("Attribuzione sorgente rifiuti spiaggiati", fontweight="bold", pad=12)
ax.set_xlabel(""); ax.set_ylabel("Total items")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.savefig(FIGS / "beach_source_attribution.png", dpi=150)
plt.show()

---
## 6. Floating Litter — Composizione materiali

In [ ]:
mat = floating["material"].value_counts()
threshold = mat.sum() * 0.02
mat_g = mat[mat >= threshold].copy()
mat_g["Other"] = mat[mat < threshold].sum()
mat_g = mat_g.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 7))
colors_pie = ["#e74c3c","#95a5a6","#3498db","#2ecc71","#f39c12","#9b59b6","#1abc9c"]
wedges, texts, autotexts = ax.pie(
    mat_g.values, labels=mat_g.index, autopct="%1.1f%%",
    colors=colors_pie[:len(mat_g)], startangle=140, pctdistance=0.82,
    wedgeprops=dict(linewidth=1.5, edgecolor="white")
)
for at in autotexts: at.set_fontsize(9)
ax.set_title("Composizione materiali — rifiuti flottanti (2021–2023)", fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig(FIGS / "floating_material_pie.png", dpi=150)
plt.show()

## 7. Floating Litter — Stagionalità

In [ ]:
month_names = ["Jan","Feb","Mar","Apr","May","Jun",
               "Jul","Aug","Sep","Oct","Nov","Dec"]
monthly = floating.groupby("month").size().reindex(range(1,13), fill_value=0)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(range(1,13), monthly.values, color="#2980b9", alpha=0.8, width=0.6)
ax.set_xticks(range(1,13)); ax.set_xticklabels(month_names)
ax.set_title("Stagionalità rifiuti flottanti — osservazioni per mese (2021–2023)",
             fontweight="bold", pad=12)
ax.set_ylabel("N° osservazioni")
for bar, val in zip(bars, monthly.values):
    if val > 0:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
                str(val), ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(FIGS / "floating_seasonality.png", dpi=150)
plt.show()

## 8. Floating Litter — Trend materiali per anno

In [ ]:
top_mats = floating["material"].value_counts().head(4).index.tolist()
mat_year = (floating[floating["material"].isin(top_mats)]
            .groupby(["year","material"]).size().unstack(fill_value=0))

mat_colors = {"Artificial polymer":"#e74c3c", "Natural matter":"#27ae60",
              "Processed wood":"#8B4513", "Food waste":"#f39c12"}

fig, ax = plt.subplots(figsize=(9, 5))
for mat_name in mat_year.columns:
    ax.plot(mat_year.index, mat_year[mat_name], marker="o", linewidth=2,
            color=mat_colors.get(mat_name, "#666"), label=mat_name)
ax.set_title("Trend materiali flottanti per anno", fontweight="bold", pad=12)
ax.set_xlabel("Year"); ax.set_ylabel("N° osservazioni")
ax.set_xticks(mat_year.index.dropna().astype(int))
ax.legend(fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.savefig(FIGS / "floating_material_trend.png", dpi=150)
plt.show()

---
## 9. Summary statistics table

In [ ]:
summary = (
    beach.groupby("beach_id")
    .agg(
        name=("beach_name","first"),
        surveys=("survey_id","count"),
        mean_items_100m=("items_per_100m","mean"),
        median_items_100m=("items_per_100m","median"),
        max_items_100m=("items_per_100m","max"),
        trend=("items_per_100m", lambda x:
               "↑" if x.iloc[-1] > x.iloc[0] else "↓" if x.iloc[-1] < x.iloc[0] else "→")
    )
    .round(1)
)
print("EU indicative threshold: 150 items/100m")
summary